In [43]:
import os
import pandas as pd
import re
import torch
from collections import defaultdict
from pathlib import Path
FOUNDATIONS = ['hyp:tag:.',
 'hyp:tag:cc',
 'hyp:tag:cd',
 'hyp:tag:dt',
 'hyp:tag:ex',
 'hyp:tag:in',
 'hyp:tag:jj',
 'hyp:tag:nn',
 'hyp:tag:nnp',
 'hyp:tag:nns',
 'hyp:tag:prp',
 'hyp:tag:prp$',
 'hyp:tag:rb',
 'hyp:tag:vb',
 'hyp:tag:vbd',
 'hyp:tag:vbg',
 'hyp:tag:vbn',
 'hyp:tag:vbp',
 'hyp:tag:vbz',
 'hyp:tok:about',
 'hyp:tok:after',
 'hyp:tok:alone',
 'hyp:tok:are',
 'hyp:tok:asleep',
 'hyp:tok:at',
 'hyp:tok:baby',
 'hyp:tok:band',
 'hyp:tok:baseball',
 'hyp:tok:beach',
 'hyp:tok:bench',
 'hyp:tok:bike',
 'hyp:tok:birthday',
 'hyp:tok:black',
 'hyp:tok:blue',
 'hyp:tok:boat',
 'hyp:tok:boy',
 'hyp:tok:boys',
 'hyp:tok:car',
 'hyp:tok:cat',
 'hyp:tok:chasing',
 'hyp:tok:child',
 'hyp:tok:children',
 'hyp:tok:concert',
 'hyp:tok:couch',
 'hyp:tok:crowd',
 'hyp:tok:dancing',
 'hyp:tok:day',
 'hyp:tok:dinner',
 'hyp:tok:dog',
 'hyp:tok:dogs',
 'hyp:tok:drinking',
 'hyp:tok:drives',
 'hyp:tok:driving',
 'hyp:tok:eating',
 'hyp:tok:eats',
 'hyp:tok:empty',
 'hyp:tok:enjoying',
 'hyp:tok:field',
 'hyp:tok:fishing',
 'hyp:tok:food',
 'hyp:tok:football',
 'hyp:tok:for',
 'hyp:tok:friends',
 'hyp:tok:from',
 'hyp:tok:game',
 'hyp:tok:girl',
 'hyp:tok:girls',
 'hyp:tok:grass',
 'hyp:tok:has',
 'hyp:tok:he',
 'hyp:tok:her',
 'hyp:tok:his',
 'hyp:tok:holding',
 'hyp:tok:horse',
 'hyp:tok:human',
 'hyp:tok:in',
 'hyp:tok:indoors',
 'hyp:tok:inside',
 'hyp:tok:is',
 'hyp:tok:kids',
 'hyp:tok:lady',
 'hyp:tok:large',
 'hyp:tok:man',
 'hyp:tok:men',
 'hyp:tok:moving',
 'hyp:tok:music',
 'hyp:tok:nap',
 'hyp:tok:near',
 'hyp:tok:new',
 'hyp:tok:no',
 'hyp:tok:nobody',
 'hyp:tok:not',
 'hyp:tok:old',
 'hyp:tok:on',
 'hyp:tok:outdoors',
 'hyp:tok:outside',
 'hyp:tok:park',
 'hyp:tok:party',
 'hyp:tok:people',
 'hyp:tok:person',
 'hyp:tok:playing',
 'hyp:tok:pool',
 'hyp:tok:race',
 'hyp:tok:racing',
 'hyp:tok:red',
 'hyp:tok:restaurant',
 'hyp:tok:riding',
 'hyp:tok:running',
 'hyp:tok:sad',
 'hyp:tok:sand',
 'hyp:tok:selling',
 'hyp:tok:shirt',
 'hyp:tok:sits',
 'hyp:tok:sitting',
 'hyp:tok:sleeping',
 'hyp:tok:smiling',
 'hyp:tok:soccer',
 'hyp:tok:something',
 'hyp:tok:stage',
 'hyp:tok:standing',
 'hyp:tok:street',
 'hyp:tok:swimming',
 'hyp:tok:talking',
 'hyp:tok:tall',
 'hyp:tok:tennis',
 'hyp:tok:their',
 'hyp:tok:there',
 'hyp:tok:to',
 'hyp:tok:train',
 'hyp:tok:tv',
 'hyp:tok:two',
 'hyp:tok:waiting',
 'hyp:tok:walking',
 'hyp:tok:watching',
 'hyp:tok:water',
 'hyp:tok:wearing',
 'hyp:tok:wet',
 'hyp:tok:while',
 'hyp:tok:white',
 'hyp:tok:with',
 'hyp:tok:woman',
 'hyp:tok:women',
 'hyp:tok:working',
 'hyp:tok:young',
 'oth:overlap:overlap25',
 'oth:overlap:overlap50',
 'oth:overlap:overlap75',
 'pre:tag:,',
 'pre:tag:.',
 'pre:tag:cc',
 'pre:tag:cd',
 'pre:tag:dt',
 'pre:tag:in',
 'pre:tag:jj',
 'pre:tag:nn',
 'pre:tag:nnp',
 'pre:tag:nns',
 'pre:tag:prp',
 'pre:tag:prp$',
 'pre:tag:rb',
 'pre:tag:vb',
 'pre:tag:vbg',
 'pre:tag:vbn',
 'pre:tag:vbp',
 'pre:tag:vbz',
 'pre:tok:air',
 'pre:tok:and',
 'pre:tok:are',
 'pre:tok:at',
 'pre:tok:baby',
 'pre:tok:ball',
 'pre:tok:band',
 'pre:tok:baseball',
 'pre:tok:basketball',
 'pre:tok:beach',
 'pre:tok:bench',
 'pre:tok:bicycle',
 'pre:tok:bike',
 'pre:tok:black',
 'pre:tok:blond',
 'pre:tok:blue',
 'pre:tok:book',
 'pre:tok:boy',
 'pre:tok:boys',
 'pre:tok:brown',
 'pre:tok:building',
 'pre:tok:car',
 'pre:tok:child',
 'pre:tok:children',
 'pre:tok:construction',
 'pre:tok:crowd',
 'pre:tok:dancing',
 'pre:tok:dirt',
 'pre:tok:dog',
 'pre:tok:dogs',
 'pre:tok:doing',
 'pre:tok:eating',
 'pre:tok:enjoying',
 'pre:tok:field',
 'pre:tok:floor',
 'pre:tok:food',
 'pre:tok:football',
 'pre:tok:for',
 'pre:tok:front',
 'pre:tok:game',
 'pre:tok:gathered',
 'pre:tok:girl',
 'pre:tok:girls',
 'pre:tok:grass',
 'pre:tok:green',
 'pre:tok:guitar',
 'pre:tok:hands',
 'pre:tok:her',
 'pre:tok:his',
 'pre:tok:holding',
 'pre:tok:horse',
 'pre:tok:in',
 'pre:tok:is',
 'pre:tok:jumps',
 'pre:tok:kitchen',
 'pre:tok:lady',
 'pre:tok:laying',
 'pre:tok:light',
 'pre:tok:little',
 'pre:tok:looking',
 'pre:tok:man',
 'pre:tok:men',
 'pre:tok:microphone',
 'pre:tok:motorcycle',
 'pre:tok:mountain',
 'pre:tok:old',
 'pre:tok:older',
 'pre:tok:on',
 'pre:tok:outside',
 'pre:tok:pants',
 'pre:tok:park',
 'pre:tok:people',
 'pre:tok:performing',
 'pre:tok:person',
 'pre:tok:player',
 'pre:tok:playing',
 'pre:tok:pool',
 'pre:tok:race',
 'pre:tok:red',
 'pre:tok:restaurant',
 'pre:tok:riding',
 'pre:tok:river',
 'pre:tok:rock',
 'pre:tok:room',
 'pre:tok:running',
 'pre:tok:runs',
 'pre:tok:sand',
 'pre:tok:shirt',
 'pre:tok:shorts',
 'pre:tok:singing',
 'pre:tok:sit',
 'pre:tok:sits',
 'pre:tok:sitting',
 'pre:tok:skateboard',
 'pre:tok:skateboarder',
 'pre:tok:small',
 'pre:tok:smiling',
 'pre:tok:snow',
 'pre:tok:snowboarder',
 'pre:tok:snowy',
 'pre:tok:soccer',
 'pre:tok:something',
 'pre:tok:stage',
 'pre:tok:standing',
 'pre:tok:store',
 'pre:tok:street',
 'pre:tok:swimming',
 'pre:tok:table',
 'pre:tok:talking',
 'pre:tok:tennis',
 'pre:tok:through',
 'pre:tok:to',
 'pre:tok:toys',
 'pre:tok:track',
 'pre:tok:train',
 'pre:tok:trees',
 'pre:tok:two',
 'pre:tok:walk',
 'pre:tok:walking',
 'pre:tok:walks',
 'pre:tok:water',
 'pre:tok:wearing',
 'pre:tok:while',
 'pre:tok:white',
 'pre:tok:with',
 'pre:tok:woman',
 'pre:tok:women',
 'pre:tok:workers',
 'pre:tok:working',
 'pre:tok:yellow',
 'pre:tok:young']
def get_indiv_concepts(formula) -> set:
    concepts = set()
    concps = re.findall(r'(?<!\bNOT\s)(?:\b(?:hyp|pre|oth):[^\s)]+)', formula)
    for c in concps:
        try:
            end_idx = c.index(')')
        except:
            end_idx = len(c)
        if c[:end_idx] in FOUNDATIONS:
            concepts.add(c[:end_idx])
 
    return concepts

def load_csv_data(filepath):
    """Load CSV and extract unit-concept mappings."""
    df = pd.read_csv(filepath)
    unit_concepts = defaultdict(set)
    
    for _, row in df.iterrows():
        unit = row['unit']
        formula = row['best_name']
        concepts = get_indiv_concepts(formula)
        unit_concepts[unit].update(concepts)
    
    return unit_concepts

def build_binary_mask(neuron_mask) -> torch.Tensor:
    num_neurons = len(neuron_mask)
    num_concepts = len(FOUNDATIONS)

    # Step 1: Initialize tensor
    tensor = torch.zeros((num_neurons, num_concepts), dtype=torch.float32)

    # Step 2: Fill in ones
    for i, concepts in enumerate(neuron_mask.values()):
        for j, concept in enumerate(FOUNDATIONS):
            if concept in concepts:
                tensor[i, j] = 1.0

    # Step 3: Compute row sums
    row_sums = tensor.sum(dim=1, keepdim=True)

    # Step 4: Normalize safely
    dist_tensor = torch.zeros_like(tensor)
    row_mask = (row_sums != 0).squeeze(1)  # True for rows with sum > 0
    dist_tensor[row_mask] = tensor[row_mask] #/ row_sums[row_mask]

    return dist_tensor

unpruned_expls = '/workspace/CCE_NLI/BERT/exp/lottery_ticket/Run0.25_3/Expls/25.0%Pruned'
neuron_formulas = defaultdict(set)
for file in os.listdir(unpruned_expls):
    try:
        
        df = pd.read_csv(os.path.join(unpruned_expls, file))
        for row in df.iterrows():
                row = row[1]
                neuron, formula, iou = row.unit, row.best_name,float("{:.3f}".format(row.best_iou))
                concepts = get_indiv_concepts(formula)
                neuron_formulas[neuron].update(concepts)
    except Exception as e:
        print(e)
        continue

mask = build_binary_mask(neuron_formulas)
col_sums = mask.sum(dim=0)

P_global = col_sums / mask.shape[0]
epsilon = 1e-12  # to avoid log(0)
        
       

'Series' object has no attribute 'unit'


In [44]:
def test_uniformity(mask):
    """Test if concepts are uniformly distributed across neurons."""
    col_sums = mask.sum(dim=0).numpy()
    
    chi2_stat, p_value = chisquare(f_obs=col_sums)
    
    print(f"Chi-Square Test for Uniformity:")
    print(f"  χ² statistic: {chi2_stat:.2f}")
    print(f"  p-value: {p_value:.10f}")  # More precision
    print(f"  Degrees of freedom: {len(col_sums) - 1}")
    print()
    print(f"  Observed range: [{col_sums.min():.2f}, {col_sums.max():.2f}]")
    print(f"  Expected (uniform): {col_sums.mean():.2f}")
    print()
    
    if p_value < 0.001:
        print(f"  EXTREMELY non-uniform (p < 0.001)")
        print(f"  Concepts are HEAVILY concentrated on certain foundations")
    elif p_value < 0.05:
        print(f"  Significantly non-uniform (p < 0.05)")
    else:
        print(f"  ✓ Cannot reject uniformity")
    
    # Show most/least common
    sorted_idx = np.argsort(col_sums)[::-1]
    print(f"\n concepts:")
    for i in range(len(sorted_idx)):
        idx = sorted_idx[i]
        print(f"    {FOUNDATIONS[idx]}: {col_sums[idx]:.1f} occurrences")
    
    
    return chi2_stat, p_value

test_uniformity(mask)

Chi-Square Test for Uniformity:
  χ² statistic: 30423.98
  p-value: 0.0000000000
  Degrees of freedom: 295

  Observed range: [1.00, 518.00]
  Expected (uniform): 22.60

  EXTREMELY non-uniform (p < 0.001)
  Concepts are HEAVILY concentrated on certain foundations

 concepts:
    pre:tag:nn: 518.0 occurrences
    hyp:tag:nn: 329.0 occurrences
    oth:overlap:overlap25: 264.0 occurrences
    hyp:tag:dt: 182.0 occurrences
    oth:overlap:overlap50: 171.0 occurrences
    hyp:tag:vbg: 147.0 occurrences
    hyp:tok:outside: 142.0 occurrences
    hyp:tag:in: 137.0 occurrences
    pre:tok:man: 131.0 occurrences
    hyp:tok:to: 126.0 occurrences
    hyp:tok:for: 123.0 occurrences
    pre:tag:jj: 120.0 occurrences
    hyp:tok:sleeping: 120.0 occurrences
    pre:tag:.: 117.0 occurrences
    hyp:tag:prp$: 111.0 occurrences
    oth:overlap:overlap75: 107.0 occurrences
    hyp:tok:man: 96.0 occurrences
    pre:tok:sitting: 95.0 occurrences
    hyp:tok:outdoors: 94.0 occurrences
    pre:tok:woman: 9

(30423.976249112624, 0.0)